# 实验九 · 伪共享：当正确的程序依然很慢

**所属**：《并行计算》第四章 · Pthread 多线程编程　|　**难度**：⭐⭐⭐⭐ 综合　|　**预计时长**：20–30 分钟

> **实验说明**
> 1. 这是本章的最后一个实验，也是视角最独特的一个。前八个实验关注的都是**正确性**与**锁**；本实验的程序**完全正确、没有数据竞争、连一把锁都不需要**，却可能比单线程还慢数倍。原因不在代码里，而在**缓存**上。
> 2. 实验对比两个版本：紧凑布局（`BadCounter`）与缓存行对齐布局（`GoodCounter`）。二者逻辑完全相同，唯一的差别是内存布局。
> 3. ⚠️ **本实验必须在多核处理器上运行。** 伪共享是多核之间的缓存一致性现象，单核不会产生这类缓存一致性开销，因而看不到效果。请在华为鲲鹏多核平台上完成本实验。
> 4. 请自上而下依次执行各单元格（Shift+Enter）。
> 5. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。

## 🎯 学习目标

完成本实验后，学生应能够：

- 说明缓存一致性以**缓存行**（而非单个变量）为单位工作，并据此定义**伪共享**
- 解释伪共享为何在无数据竞争、无锁的情况下仍会严重拖慢程序
- 用 MESI 协议描述「缓存行乒乓」的产生过程
- 使用**缓存行填充**与 `_Alignas` 消除伪共享，并说明为何填充与对齐缺一不可
- 说明 `aligned_alloc` 在动态分配时的必要性
- 理解 `volatile` 在本实验中为何不可省略——否则编译器会使测量失去意义

## 🗺️ 学习路径

1. **准备阶段**：回顾缓存行与 MESI 协议，为理解伪共享打基础
2. **概念建立**：区分「逻辑共享」与「物理共享」，给出伪共享的定义
3. **版本一（紧凑布局）**：多个计数器位于同一缓存行，观察多核下的性能急剧下降
4. **机器级验证**：用 `objdump` 确认 `volatile` 使每次迭代都真正访存
5. **版本二（对齐布局）**：填充 + `_Alignas` 让每个计数器独占缓存行
6. **性能对比**：随线程数增长，观察两个版本的分化——一个负加速比，一个接近线性

## 1. 背景：从「线程与锁」到「缓存与硬件」

本章的前八个实验，围绕两个主题展开：

- **正确性**：数据竞争、死锁、时序错误，以及消除它们的各种同步工具；
- **锁的粒度**：临界区多大、用哪种锁，权衡并发度与开销。

这些讨论都停留在「线程」与「锁」的层面。本实验要揭示一个更底层、也更隐蔽的问题：

> 一段**完全正确、没有任何数据竞争、连锁都不需要**的并行代码，
> 可能仅仅因为两个线程的变量恰好落在**同一条缓存行**上，
> 就慢到接近串行，甚至出现**负加速比**——线程越多越慢。

这个问题在源代码里**完全看不出来**。它不违反任何语言规则，编译器不会警告，逻辑推理也挑不出错。它只在真实多核硬件上运行时才暴露，根源在于处理器的**缓存一致性机制**。

理解它，是并发编程「最后一公里」——写出不仅正确、而且真正高效的并行代码。这也是本章选择深入到 Pthreads 与硬件层面的意义。

## 2. 回顾：缓存行与 MESI 协议

（本节内容在第二章已详细介绍，这里简要回顾，作为理解伪共享的基础。）

### 2.1 缓存行

CPU 与内存之间传输数据的最小单位**不是字节，而是一整条缓存行**（cache line）。主流架构（含鲲鹏所属的 ARMv8，以及 x86）的缓存行通常是 **64 字节**。

这意味着：读取一个 8 字节的 `long`，硬件实际搬运的是它所在的**整条 64 字节缓存行**。一条缓存行可以容纳 8 个 `long`。

### 2.2 MESI 一致性协议

多核系统中，每个核心有自己私有的 L1／L2 缓存。当同一条缓存行的副本出现在多个核心的私有缓存中时，必须保证它们看到的数据一致——这由 **MESI 协议**维护。缓存行的每个副本处于四种状态之一：

<!--
| 状态 | 含义 |
|---|---|
| **M**（Modified，已修改） | 本核已改写，与主存不一致，且是唯一有效副本 |
| **E**（Exclusive，独占） | 只有本核持有，与主存一致 |
| **S**（Shared，共享） | 多个核心同时持有，均与主存一致 |
| **I**（Invalid，无效） | 本核副本已失效，再次访问必须重新获取 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">状态</th>
      <th style="text-align: left;">含义</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>M</strong>（Modified，已修改）</td>
      <td style="text-align: left;">本核已改写，与主存不一致，且是唯一有效副本</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>E</strong>（Exclusive，独占）</td>
      <td style="text-align: left;">只有本核持有，与主存一致</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>S</strong>（Shared，共享）</td>
      <td style="text-align: left;">多个核心同时持有，均与主存一致</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>I</strong>（Invalid，无效）</td>
      <td style="text-align: left;">本核副本已失效，再次访问必须重新获取</td>
    </tr>
  </tbody>
</table>

**关键规则**：任何一个核心要**写**某条缓存行，必须先把其他核心中该行的所有副本置为 **Invalid**。这一步会产生**跨核的一致性通信**。

记住这条规则——伪共享的全部代价都来源于它。

## 3. 伪共享：逻辑无关，物理冲突

考虑一个简单的并行计数任务：$t$ 个线程，每个线程反复给**自己专属**的一个计数器加一。

```c
typedef struct {
  long value;        // 仅 8 字节
} BadCounter;

BadCounter counters[MAX_THREADS];

void *Worker(void *rank) {
  long id = (long)rank;
  for (long i = 0; i < N; ++i) {
    counters[id].value++;      // 每个线程只碰自己的 counters[id]
  }
  return NULL;
}
```

**这段代码在逻辑上毫无问题**：

- 每个线程只访问 `counters[id]`，各写各的，互不重叠；
- 不存在数据竞争（对照实验三的定义，不满足「访问同一位置」）；
- 因此**不需要任何锁**，结果永远正确。

然而，在多核上运行，它可能比单线程版本还要慢数倍。

### 问题所在：8 个计数器位于同一条缓存行

`sizeof(BadCounter)` 只有 8 字节，因此 `counters[0]` 到 `counters[7]` 这 8 个元素**恰好落在同一条 64 字节缓存行内**：

```
   ┌──────┬──────┬──────┬──────┬──────┬──────┬──────┬──────┐
   │ [0]  │ [1]  │ [2]  │ [3]  │ [4]  │ [5]  │ [6]  │ [7]  │
   └──────┴──────┴──────┴──────┴──────┴──────┴──────┴──────┘
     T0     T1     T2     T3     T4     T5     T6     T7
   │←──────────────────  一条 64 字节缓存行  ──────────────────→│
```

现在回想 2.2 节那条 MESI 规则：**线程 0 写 `counters[0]`，会把整条缓存行置为 M 状态，从而使其余 7 个核心中该行的副本全部失效**。于是线程 1～7 下次访问自己的计数器时，即使它们的数据根本没被改动，也必须重新从上级缓存获取整条缓存行。

> **「伪」共享之名由此而来**：线程之间在逻辑上并未共享任何数据，却在物理上共享了同一条缓存行。

### ⚠️ 这不是数据竞争，加锁只会更糟

必须强调：伪共享**不是**数据竞争。数据竞争是多个线程访问**同一个变量**且至少一个写；而这里每个线程访问的是**不同的变量**，只是它们碰巧在同一条缓存行上。

因此，加锁**不能**解决伪共享——恰恰相反，加锁会引入更多对同一缓存行的争抢，使情况更糟。伪共享必须从**数据布局**上根治。

## 4. 缓存行乒乓

多个核心轮流写入同一条缓存行时，这条缓存行便在各核的私有缓存之间被反复「争夺」——每一次写入都触发一轮跨核的一致性通信。这一现象称为**缓存行乒乓**（cache line ping-pong）。

```
      核心 0 私有 L1            核心 1 私有 L1
     ┌──────────────┐         ┌──────────────┐
     │ 缓存行：M     │  ◄────► │ 缓存行：I     │
     └──────────────┘  一致性  └──────────────┘
                        通信
   核心 1 要写自己的 counters[1]，必须先把整条缓存行从核心 0 抢回来，
   并把核心 0 的副本置为 I。下一刻核心 0 又要抢回去。如此往复。
```

它带来三重后果：

**① 延迟放大。** 本应命中 L1 缓存的访问（约 4 个周期），退化为跨核的一致性事务（数十至上百个周期）。

**② 线程越多越慢。** 参与争抢的核心越多，一致性通信量呈**超线性**增长，可能出现**负加速比**——8 个线程比 1 个线程还慢。

**③ 加锁无济于事。** 这不是数据竞争，加锁只会让对缓存行的争抢更激烈。必须从数据布局上根治。

> 这正是本实验最反直觉之处：程序完全正确、没有锁、每个线程只碰自己的数据，性能却因为一个纯粹的**内存布局**问题而大幅劣化。

## 5. 环境准备

In [ ]:
import platform, subprocess, shutil, sys, os, re

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
NCPU = os.cpu_count()
print("CPU 核心:", NCPU)

if CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
elif NCPU == 1:
    print("\n" + "=" * 64)
    print("⚠️  严重提示：本机只有 1 个核心，无法演示伪共享！")
    print("=" * 64)
    print("伪共享是多核之间的缓存一致性现象。单核系统中同一时刻只有一个")
    print("线程在运行，不存在跨核的缓存行争抢，因此两个版本耗时会基本相同。")
    print("这不是程序错误，而是原理使然。")
    print("请务必在华为鲲鹏等多核平台上重做本实验，方能观察到真实效果。")
else:
    print(f"\n✅ 环境就绪：{NCPU} 核可用。伪共享需要多核才能显现，本机满足条件。")
    print("   建议将线程数设到接近核心数，效果最明显。")


### 编译与运行工具函数

本实验只有一个源文件。编译选项与全章一致；`-O3` 在这里尤其关键——伪共享是一种在充分优化下才凸显的微架构效应。

In [2]:
SRC_DIR = "src_falsesharing"
os.makedirs(SRC_DIR, exist_ok=True)


def compile_c(src, out):
    """用全章统一选项编译一个源文件，成功返回可执行文件名，失败返回 None。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    cmd = f"{base} -O3 -fPIC -pthread -Wall -Wextra {src} -o {out} -lpthread -lm"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print("✅ 编译成功：", cmd)
        if r.stderr.strip():
            print(r.stderr.strip())
        return out
    print("❌ 编译失败：\n", r.stderr)
    return None


def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, r.stdout + r.stderr


def run_bin(out, *args, echo=True, timeout=300):
    """运行可执行文件并返回其标准输出；echo=True 时同时打印。"""
    r = subprocess.run(
        ["./" + out] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if echo:
        print(r.stdout, end="")
        if r.returncode != 0 and r.stderr:
            print("STDERR:", r.stderr)
    return r.stdout


def parse_times(text):
    """解析输出，返回 (bad_ms, good_ms, speedup)。"""
    b = re.search(r"Bad.*?([\d.]+)\s*$", text, re.M)
    g = re.search(r"Good.*?([\d.]+)\s*$", text, re.M)
    s = re.search(r"Speedup.*?([\d.]+)x", text)
    return (
        float(b.group(1)) if b else None,
        float(g.group(1)) if g else None,
        float(s.group(1)) if s else None,
    )


## 6. 两种布局的设计

本实验的程序在同一次运行中对比两个版本，它们的**计算完全相同**——每个线程给自己的计数器累加 `iterations` 次——唯一的差别是计数器的内存布局。

### 6.1 紧凑布局（`BadCounter`）

```c
typedef struct {
  volatile long value;      // 8 字节
} BadCounter;
```

`sizeof(BadCounter)` 为 8，因此 8 个计数器共享一条 64 字节缓存行——这正是第 3 节所述伪共享的根源。

### 6.2 对齐布局（`GoodCounter`）

```c
#define CACHE_LINE_SIZE 64

typedef struct {
  _Alignas(CACHE_LINE_SIZE) volatile long value;
  char padding[CACHE_LINE_SIZE - sizeof(long)];   // 填充到整条缓存行
} GoodCounter;
```

填充与对齐分别解决两个不同层面的问题：

- **填充**（`padding`）把结构体撑大到 64 字节，保证任意两个相邻元素 `arr[i]` 与 `arr[i+1]` 之间至少相隔一条缓存行；
- **`_Alignas(64)`** 保证每个元素从缓存行边界开始，即约束数组的起始位置。

> ⚠️ **为什么填充还不够，必须加 `_Alignas`？**
>
> 填充只约束元素**之间**的间距，并不保证数组的**首地址**落在缓存行边界上。设想数组从地址 16 开始：`arr[0]` 占据地址区间 [16, 80)，横跨缓存行 {0–64} 与 {64–128}；`arr[1]` 占据 [80, 144)，横跨 {64–128} 与 {128–192}。此时 `arr[0]` 与 `arr[1]` **仍然共享缓存行 {64–128}**，写这两个计数器依旧会发生伪共享。
>
> 因此，首地址未对齐时，填充只能**削弱**伪共享（从「8 个线程争抢 1 条缓存行」降为「相邻 2 个线程共享 1 条边界缓存行」），而不能将其根除。`_Alignas(64)` 正是用来消除这一残余，使每个元素独占一条完整的缓存行。

### 6.3 对齐由谁保证：`_Alignas` 与 `aligned_alloc` 的分工

`_Alignas` 能否单独保证首地址对齐，取决于内存**来自何处**。这是一个容易混淆、却必须分清的要点。

**情形一：栈上或静态数组。** 若数组这样定义：

```c
GoodCounter counters[MAX_THREADS];   // 栈上或静态存储
```

则 `_Alignas` **单独即可**——编译器在为该数组分配存储时，会遵循类型的对齐要求，保证整个数组按 64 字节对齐。此时无需 `aligned_alloc`。

**情形二：堆上动态分配。** 若数组由 `malloc` 在堆上分配，情况就不同了：

```c
GoodCounter *counters = malloc(n * sizeof(GoodCounter));   // 首地址通常只按 16 字节对齐
```

`malloc` 只保证满足**基本类型**的对齐（在多数平台上为 16 字节），它**不理会**类型声明中的 `_Alignas(64)`。因此，动态分配时必须改用 C11 的 `aligned_alloc`，显式指定 64 字节对齐。本实验的做法即是如此：

```c
good_counters = aligned_alloc(
    CACHE_LINE_SIZE,
    round_up_to_line((size_t)thread_count * sizeof(GoodCounter)));
```

`aligned_alloc` 要求分配的字节数是对齐值的整数倍，因此程序用 `round_up_to_line` 把大小向上取整到整条缓存行的倍数。

> **本实验的计数器数组是动态分配的**，因此 `_Alignas` 与 `aligned_alloc` 二者缺一不可：
> - `_Alignas`（连同填充）约束**类型的内部布局**，保证相邻元素相隔一条缓存行；
> - `aligned_alloc` 约束**这块堆内存的起始地址**，保证第一个元素就落在缓存行边界。
>
> 若仅用 `_Alignas` 而以 `malloc` 分配，首地址便可能不对齐，6.2 节所述的残余伪共享就会出现。

### 6.4 ⚠️ `volatile` 为何不可省略

源码中两个计数器的 `value` 都标了 `volatile`。**注意：这是本实验成立的前提**。

考虑没有 `volatile` 的情形。循环体是：

```c
for (long i = 0; i < iterations; ++i) {
  counters[id].value++;      // 读—改—写
}
```

在 `-O3` 下，编译器会发现：循环期间没有别的代码读取 `counters[id].value`，于是它可以把这个变量**留在寄存器里累加**，不必每次都从内存读取——具体能省去多少访存，取决于编译器与目标架构，某些平台甚至会把整个循环塌缩为「结束后写回一次」。这样一来：

- 循环体不再是源码所写的那个「完整的读—改—写」；
- 每次迭代实际发生的内存访问被大幅削减；
- 缓存行的争抢随之减弱甚至消失——伪共享被「优化」掉了，但不是因为代码正确，而是因为要测量的工作根本没在做。

此时对比两个版本，测的已不是我们想考察的工作，结果毫无意义。

`volatile` 的作用正是**强制每次迭代都真正读、真正写内存**，使两个版本都执行完整的 `iterations` 次内存写入。这样，二者之间唯一的差别才回归到内存布局本身。第 8 节将用 `objdump` 直接验证这一点。

> 这与实验五中忙等待对 `volatile` 的依赖同源：都是为了阻止编译器把「反复访存」优化成「只访存一次」。区别在于，实验五靠它保证**正确性**，本实验靠它保证**测量有效**。

## 7. 完整源码与运行

In [ ]:
%%writefile {SRC_DIR}/pthread_false_sharing.c
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define MAX_THREADS 64
#define CACHE_LINE_SIZE 64  // 64 bytes on ARMv8 (Kunpeng) and on x86

// volatile is essential to this experiment, not a style choice. Without it,
// -O3 is free to keep the counter in a register for the whole loop and write
// it back once, collapsing N increments into a single add. The loop would then
// perform no repeated stores, no cache line would ever be contended, and the
// measurement would compare two programs that do not do the work being timed.
// volatile forces one load and one store per iteration in BOTH versions, so
// the only remaining difference between them is the memory layout.
// (Verify with: objdump -d --disassemble=Worker_bad ./false_sharing)

// Compact layout: sizeof(BadCounter) is 8, so eight counters share one 64-byte
// cache line. Each thread writes only its own element, so there is no data
// race and no lock is needed. The cores still fight over the line, because
// coherence works at cache-line granularity, not at variable granularity.
typedef struct {
  volatile long value;
} BadCounter;

// Padded and aligned layout. Padding alone only makes the struct 64 bytes
// wide; it does not guarantee that element 0 begins at a cache-line boundary.
// _Alignas forces that boundary, and the allocation must be aligned as well,
// otherwise the array can straddle lines and two threads can still collide.
typedef struct {
  _Alignas(CACHE_LINE_SIZE) volatile long value;
  char padding[CACHE_LINE_SIZE - sizeof(long)];
} GoodCounter;

int thread_count = 0;
long iterations = 0;

BadCounter *bad_counters = NULL;
GoodCounter *good_counters = NULL;

static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

void *Worker_bad(void *rank) {
  long my_rank = (long)rank;
  for (long i = 0; i < iterations; ++i) {
    bad_counters[my_rank].value++;
  }
  return NULL;
}

void *Worker_good(void *rank) {
  long my_rank = (long)rank;
  for (long i = 0; i < iterations; ++i) {
    good_counters[my_rank].value++;
  }
  return NULL;
}

// Runs one version and returns its wall time in milliseconds. Only the array
// that this version uses is reset, so both results survive for the final check.
static double run_version(void *(*worker)(void *), int is_bad) {
  pthread_t *thread_handles = malloc(thread_count * sizeof(pthread_t));
  if (thread_handles == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    exit(1);
  }

  for (int i = 0; i < thread_count; ++i) {
    if (is_bad) {
      bad_counters[i].value = 0;
    } else {
      good_counters[i].value = 0;
    }
  }

  double start = get_time_ms();
  for (long t = 0; t < thread_count; ++t) {
    if (pthread_create(&thread_handles[t], NULL, worker, (void *)t) != 0) {
      fprintf(stderr, "Error: pthread_create failed\n");
      exit(1);
    }
  }
  for (long t = 0; t < thread_count; ++t) pthread_join(thread_handles[t], NULL);
  double elapsed = get_time_ms() - start;

  free(thread_handles);
  return elapsed;
}

// Both versions must produce exactly the same, correct counters. The point of
// the experiment is that they are equally correct and differ only in speed.
static const char *check_diff(long expected) {
  for (int i = 0; i < thread_count; ++i) {
    if (bad_counters[i].value != expected ||
        good_counters[i].value != expected) {
      return "FAIL";
    }
  }
  return "PASS";
}

// Rounds size up to a whole number of cache lines, as aligned_alloc requires
// the size to be a multiple of the alignment.
static size_t round_up_to_line(size_t bytes) {
  return ((bytes + CACHE_LINE_SIZE - 1) / CACHE_LINE_SIZE) * CACHE_LINE_SIZE;
}

int main(int argc, char *argv[]) {
  if (argc != 3) {
    fprintf(stderr, "Usage: %s <thread_count> <iterations>\n", argv[0]);
    return 1;
  }

  thread_count = (int)strtol(argv[1], NULL, 10);
  iterations = strtol(argv[2], NULL, 10);

  if (thread_count <= 0 || thread_count > MAX_THREADS) {
    fprintf(stderr, "Error: thread_count must be between 1 and %d\n",
            MAX_THREADS);
    return 1;
  }
  if (iterations <= 0) {
    fprintf(stderr, "Error: iterations must be positive\n");
    return 1;
  }

  bad_counters = aligned_alloc(
      CACHE_LINE_SIZE,
      round_up_to_line((size_t)thread_count * sizeof(BadCounter)));
  good_counters = aligned_alloc(
      CACHE_LINE_SIZE,
      round_up_to_line((size_t)thread_count * sizeof(GoodCounter)));
  if (bad_counters == NULL || good_counters == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }

  printf("False Sharing: compact vs cache-line aligned counters\n");
  printf("Threads: %d, Iterations per thread: %ld\n", thread_count, iterations);
  printf(
      "sizeof(BadCounter)  = %zu bytes -> %zu of them share one cache line\n",
      sizeof(BadCounter), (size_t)CACHE_LINE_SIZE / sizeof(BadCounter));
  printf("sizeof(GoodCounter) = %zu bytes -> one per cache line\n\n",
         sizeof(GoodCounter));

  double time_bad = run_version(Worker_bad, 1);
  double time_good = run_version(Worker_good, 0);

  printf("%-34s %12s\n", "Version", "Time (ms)");
  printf("%-34s %12.3f\n", "Bad  (compact, false sharing)", time_bad);
  printf("%-34s %12.3f\n", "Good (padded + _Alignas 64)", time_good);
  printf("\nSpeedup from removing false sharing: %.2fx\n",
         time_bad / time_good);
  printf("Check (both counters correct): %s\n", check_diff(iterations));
  printf(
      "\nBoth versions are race-free and equally correct; only the memory "
      "layout differs.\n");
  printf(
      "A single core generates no coherence traffic, so the gap appears only "
      "on a real multi-core machine.\n");

  free(bad_counters);
  free(good_counters);
  return 0;
}

In [ ]:
fs = compile_c(f"{SRC_DIR}/pthread_false_sharing.c", f"{SRC_DIR}/pthread_false_sharing")
print()
NT = max(2, min(8, os.cpu_count()))
out = run_bin(fs, NT, 50_000_000)

### 首轮观察

输出开头印证了两种布局的大小差异：`BadCounter` 为 8 字节（8 个共处一条缓存行），`GoodCounter` 为 64 字节（一个独占一条缓存行）。

若在**多核**上运行，紧凑版本（Bad）的耗时应明显高于对齐版本（Good），加速比可能达到数倍。而末尾的 `Check: PASS` 确认了一个关键事实：**两个版本的计算结果完全相同、同样正确**——它们的差别纯粹在速度，而速度差纯粹来自内存布局。

若在单核上运行，两者耗时基本持平，看不出差距（原因见第 5 节的提示）。

## 8. 机器级验证：`volatile` 确实强制了每次访存

6.4 节断言：没有 `volatile`，编译器会把整个累加循环塌缩成一次写回。下面构造一个去掉 `volatile` 的副本，用 `objdump` 对比二者的循环体加以验证。此副本**仅用于反汇编，不参与计时**。

In [ ]:
# 构造去掉 volatile 的副本，仅用于反汇编对比，不参与计时
src = open(f"{SRC_DIR}/pthread_false_sharing.c").read()
novol = src.replace("volatile long value", "long value")
assert novol != src, "未找到 volatile 声明"
open(f"{SRC_DIR}/pthread_false_sharing_novolatile.c", "w").write(novol)

no_vol = compile_c(
    f"{SRC_DIR}/pthread_false_sharing_novolatile.c",
    f"{SRC_DIR}/pthread_false_sharing_novolatile",
)
print()


def loop_body(binary, func):
    """提取函数最内层循环的指令，依据第一条向后跳转判定。"""
    rc, asm = sh(f"objdump -d {binary} --disassemble={func}")
    lines = [l.rstrip() for l in asm.split("\n") if re.match(r"^\s+[0-9a-f]+:", l)]
    addr = lambda l: int(re.match(r"^\s+([0-9a-f]+):", l).group(1), 16)
    for i, l in enumerate(lines):
        m = re.search(
            r"\b(?:jg|jl|jge|jle|jne|je|jnz|jz|bne|beq|"
            r"cbnz|cbz|b\.\w+)\s+([0-9a-f]+)",
            l,
        )
        if m and int(m.group(1), 16) < addr(l):
            t = int(m.group(1), 16)
            return [x for x in lines if t <= addr(x) <= addr(l)]
    return []


def mnemonic(line):
    """从 objdump 的一行中稳健地提取指令助记符。

    objdump 的行格式为『地址: <TAB> 机器码 <TAB> 助记符 <TAB> 操作数』，
    不同架构/版本的制表符分段数可能不同（AArch64 常为 4 段），
    因此不能简单取某一段，而应跳过『地址』与纯十六进制的『机器码』，
    取第一个含字母的 token 作为助记符。
    """
    after = re.sub(r"^\s*[0-9a-f]+:\s*", "", line)  # 去掉行首『地址:』
    for tok in re.split(r"[\s\t]+", after):
        if not tok:
            continue
        if re.fullmatch(r"[0-9a-f]{2,}", tok):  # 纯十六进制 = 机器码，跳过
            continue
        return tok  # 第一个非机器码 token = 助记符
    return ""


def mem_ops(body):
    """统计循环体内的读/写内存指令，兼容 AArch64 与 x86-64。

    注意：AArch64 的『读』会包含读取循环上界（iterations）的那条 ldr，
    因此读的计数可能比对 counter 的实际读取多 1。判断伪共享是否成立，
    关键看『写内存』——只要每次迭代都有一条 str/mov 写回，写内存就会
    触发缓存一致性动作。
    """
    loads = stores = 0
    for l in body:
        mnem = mnemonic(l)
        txt = l
        if "(%rip)" in txt:  # x86 常量池访问，跳过
            continue
        # ---- AArch64：load = ldr/ldp/ldur…，store = str/stp/stur… ----
        if re.match(r"ld[rpu]", mnem):
            loads += 1
            continue
        if re.match(r"st[rpu]", mnem):
            stores += 1
            continue
        # ---- x86-64：mov (%reg),%reg = 读；mov %reg,(%reg) = 写 ----
        if mnem.startswith("mov"):
            if re.search(r"\([^)]*%\w+[^)]*\),\s*%\w+", txt):
                loads += 1
            elif re.search(r"%\w+,\s*\([^)]*%\w+[^)]*\)", txt):
                stores += 1
    return loads, stores


for binary, label in [(fs, "有 volatile"), (no_vol, "无 volatile")]:
    body = loop_body(binary, "Worker_bad")
    loads, stores = mem_ops(body)
    print(f"═══ Worker_bad · {label} ═══")
    print(f"  循环体 {len(body)} 条指令，其中 读内存 {loads} 处，写内存 {stores} 处")
    for l in body:
        print("    " + re.sub(r"^\s+", "", l)[:70])
    print()


### 💡 验证结果的解读

关注两个版本 `Worker_bad` 循环体内的读内存(`ldr`/`mov` 读)与写内存(`str`/`mov` 写)指令。

> **说明**：AArch64 上「读内存」的计数会**多算一条**——除了读取 counter，循环每轮还要 `ldr` 读一次循环上界 `iterations`。因此判断伪共享是否成立，**关键看「写内存」**：只要每次迭代都有一条 `str` 写回 counter，写操作就会触发缓存一致性动作。

**有 `volatile`**（AArch64 示例）：
```
ldr x0, [x2] ← 读 counter 当前值
add x0, x0, #1 ← 加一
str x0, [x2] ← 写回 counter ★ 每次迭代都写内存
```
循环体是完整的「读—改—写」，与源码 `counters[id].value++` 语义一致。

**无 `volatile`**（AArch64 示例）：
```
add x0, x3, x1 ← 在寄存器中算出新值（不再 ldr 读 counter）
str x0, [x2] ← 仍然写回 counter ★ 写内存指令保留
```
编译器把 counter 的**读**提出了循环（值在寄存器中推算），但**每次迭代仍保留 `str` 写回**。

> **这正是本实验能在 AArch64（鲲鹏、ARM 开发板）上正常显现伪共享的原因**：无论有无 `volatile`，循环体每次迭代都执行 `str` 写内存，而伪共享正是由**跨核的写**触发的。
>
> 但请注意，「写回是否保留」依赖编译器与架构：在**某些编译器/架构组合**下，若累加结果无人观测，整个循环可能被「死代码消除」——对照版（Good）被优化成空循环，测出的加速比便是**假象**。判定方法是「**迭代翻倍**」：真实执行的循环，耗时应随迭代数线性增长；若迭代数翻倍而耗时不变，说明该循环已被消除，对比无效。

`volatile` 的作用，是**禁止编译器对访存做上述省略**，保证两个版本都执行完整的 `iterations` 次内存事务，使二者唯一的差别回归到内存布局本身。

> 这与实验五中忙等待对 `volatile` 的依赖同源：都是为了阻止编译器把「反复访存」优化掉。区别在于，实验五靠它保证**正确性**，本实验靠它保证**测量有效**。

## 9. 性能对比：线程数越多，差距越大

伪共享最鲜明的特征是：**参与争抢的核心越多，代价越高**。下面固定每线程的工作量，改变线程数，观察两个版本的分化。

In [ ]:
import matplotlib.pyplot as plt

ITERS = 50_000_000
max_t = min(8, max(2, os.cpu_count()))
thread_list = [t for t in (1, 2, 4, 8) if t <= max_t]
if max_t not in thread_list:
    thread_list.append(max_t)
    thread_list = sorted(set(thread_list))

bad_ms, good_ms, speedups = [], [], []
print(f"每线程迭代 {ITERS:,} 次，{os.cpu_count()} 核\n")
print(f"{'线程数':>8}{'紧凑Bad(ms)':>14}{'对齐Good(ms)':>16}{'Bad/Good':>12}")
print("-" * 52)
for t in thread_list:
    b, g, s = parse_times(run_bin(fs, t, ITERS, echo=False))
    bad_ms.append(b)
    good_ms.append(g)
    speedups.append(s)
    print(f"{t:>8}{b:>14.1f}{g:>16.1f}{s:>11.2f}x")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))

ax1.plot(
    thread_list,
    bad_ms,
    "o-",
    label="Compact layout (false sharing)",
    color="#C7000B",
    lw=2,
    ms=7,
)
ax1.plot(
    thread_list,
    good_ms,
    "s-",
    label="Padded layout (no false sharing)",
    color="#2E7D32",
    lw=2,
    ms=7,
)
ax1.set_xlabel("Threads")
ax1.set_ylabel("Time (ms, lower is better)")
ax1.set_title("Time of the two layouts")
ax1.set_xticks(thread_list)
ax1.legend()
ax1.grid(alpha=0.3)

ax2.bar([str(t) for t in thread_list], speedups, color="#295E96")
ax2.axhline(1.0, ls="--", c="gray", lw=1.2)
for i, s in enumerate(speedups):
    ax2.text(i, s, f"{s:.1f}x", ha="center", va="bottom", fontsize=9)
ax2.set_xlabel("Threads")
ax2.set_ylabel("Bad / Good time ratio")
ax2.set_title("Speedup from removing false sharing")
ax2.grid(axis="y", alpha=0.3)

fig.suptitle(f"Cost of false sharing vs thread count ({os.cpu_count()} cores)")
plt.tight_layout()
plt.show()

if os.cpu_count() == 1:
    print("\n⚠️  本机为单核，两个版本耗时基本持平属于预期，伪共享无法显现。")
    print("    请在鲲鹏多核平台上重跑本单元，观察 Bad 版本随线程数增加而急剧变慢。")
else:
    print(f"\n随线程数增加，紧凑布局的耗时因缓存行争抢而上升，")
    print("对齐布局则接近理想扩展。二者的差距（Bad/Good）随线程数拉大。")


### 结果解读

在多核平台上，预期会观察到：

**① 紧凑布局（Bad）随线程数增加而变慢，甚至出现负加速比。** 线程越多，争抢同一条缓存行的核心越多，一致性通信量超线性增长。8 个线程时，它可能比单线程还慢——这就是**负加速比**。

**② 对齐布局（Good）接近线性扩展。** 每个计数器独占一条缓存行，核心之间不再产生一致性通信，各线程互不干扰。

**③ 二者的差距随线程数拉大。** 单线程时两者持平（无跨核争抢），线程越多，差距越显著。

**④ 两个版本的结果完全相同、同样正确。** 这是本实验最需要记住的一点：**它们的差别不在正确性，而纯在性能；性能差又纯粹来自内存布局**。

### 空间换带宽的代价

对齐布局的代价是**内存占用**：每个计数器从 8 字节膨胀到 64 字节，增大到 8 倍。但在多核场景下，这笔「用空间换带宽」的交易几乎总是划算的——几百字节的额外内存，换来数倍的性能提升。

### 如何在真实平台上取证

讲义建议用 `perf` 直接观测缓存失效，证据比耗时更直观：

```bash
perf stat -e cache-misses,cache-references ./false_sharing 8 50000000
```

对比紧凑版与对齐版的 `cache-misses` 计数，紧凑版会高出很多——这些多出的失效，正是缓存行乒乓造成的。

## 10. 结果分析

本实验把并发的视角从「线程与锁」下探到「缓存与硬件」，建立了三项认识：

**① 正确不等于高效。** 一段没有数据竞争、不需要锁、逻辑完全正确的代码，可能因为一个纯粹的内存布局问题而丧失全部并行收益。正确性与性能是两个独立的维度，都需要专门对待。

**② 硬件的工作单位与程序的逻辑单位不一致。** 程序员眼中的逻辑单位是「变量」，而缓存一致性的工作单位是「缓存行」。伪共享正源于这一错位——逻辑上无关的变量，物理上被绑在了同一条缓存行里。理解并发性能，必须理解硬件实际如何搬运数据。

**③ 有些问题加锁解决不了，只能改数据布局。** 伪共享不是同步问题，任何锁都无法消除它，反而会加剧争抢。它的唯一解法是让互相独立的数据落在不同的缓存行上。

### 🎓 结论：并发性能的最后一公里

本章从「能并行」出发，历经「正确并行」（数据竞争、死锁、各种同步原语），最终抵达「高效并行」。本实验揭示的伪共享，正是「高效并行」中最隐蔽的一环：

> 它不在源代码里，不违反任何语言规则，逻辑推理挑不出错，编译器不会警告。
> 它只在真实多核硬件上运行时才暴露，根源在处理器的缓存一致性机制。

这也解释了本章为何选择深入到 Pthreads 与硬件层面，而非停留在更高层的封装：**只有理解了线程、锁与缓存实际如何运作，才能既写出正确的并发程序，又让它在多核上真正跑得快。**

至此，本章「一条主线、六个模块」的旅程走完：从进程与线程，到数据竞争与临界区，到互斥量与死锁，到信号量与生产者-消费者，到条件变量与屏障，最终到微架构感知的伪共享。**先保证正确，再追求高效**——这条贯穿始终的原则，是并发编程最根本的方法论。

## 11. 🔧 动手练习

请在**多核平台**上修改代码、重新编译并运行，观察行为的变化：

1. 把线程数从 1 逐步增加到核心数以上（如 1、2、4、8、16），记录紧凑版本的耗时，找出它由「随线程数变快」转为「随线程数变慢」的拐点，并解释该拐点与核心数的关系。
2. 去掉两个计数器的 `volatile`，重新编译运行，观察两个版本的耗时是否都大幅下降且趋于相同。结合第 8 节的反汇编，解释这一现象，并说明为什么此时的测量没有意义。
3. 修改 `GoodCounter`，把填充大小依次改为 8、16、32、64 字节，测量各自的耗时，找出在本机上消除伪共享所需的最小填充大小，并推断本机的缓存行大小。
4. 用 `perf stat -e cache-misses ./false_sharing 8 50000000` 分别测量紧凑版与对齐版的缓存失效次数，验证紧凑版的失效次数远高于对齐版。
5. 保持紧凑布局，但让每个线程只写 `counters[id * 8]`（间隔 8 个 `long`，即隔开一条缓存行），观察性能是否恢复。解释这种「手动拉开间距」的做法与 `_Alignas` 填充的异同。

## 12. 🤔 思考题

- 伪共享与数据竞争都表现为「多个线程、同一块内存区域、性能或正确性出问题」。请从**发生的层次**（逻辑 vs 物理）和**解决的手段**（同步 vs 布局）两方面，说明二者的根本区别。
- 为什么加锁不能解决伪共享，反而会使情况更糟？请结合 MESI 协议中「写操作使其他副本失效」这条规则来分析。
- 对齐布局用 8 倍的内存换取性能。在什么情况下这笔交易**不**划算？如果计数器有几百万个而非几个，还应该给每个都填充到一条缓存行吗？
- `_Alignas` 保证结构体类型的对齐，`aligned_alloc` 保证动态分配的起始地址对齐。如果只用了 `_Alignas` 而用普通 `malloc` 分配数组，可能出现什么问题？为什么栈上的数组（如 `GoodCounter arr[8];`）通常不需要 `aligned_alloc`？
- 伪共享出现在多个线程**写**同一条缓存行时。如果所有线程都只**读**同一条缓存行（没有任何写），还会有性能问题吗？请结合 MESI 的 Shared 状态说明。
- 现代 C++ 提供了 `std::hardware_destructive_interference_size` 来获取缓存行大小，而本实验把它硬编码为 64。在一个缓存行大小可能是 32、64 或 128 字节的异构环境中，硬编码 64 会带来什么问题？应如何稳妥地处理？

## 13. 全章总结

本实验是第四章的收官。借此机会，回顾整章「一条主线、六个模块」的完整脉络：

<!--
| 模块 | 主题 | 核心实验 | 关键认识 |
|---|---|---|---|
| 一 | 共享内存并行与 Pthread 基础 | Hello、矩阵向量乘 | Fork-Join、传参规范、公平的性能测量 |
| 二 | 数据竞争与临界区 | π 估算、消息传递 | 数据竞争的定义、时序错误、`volatile` |
| 三 | 互斥量：正确性与锁粒度 | π 估算、哲学家就餐 | 最小化临界区、死锁四条件、资源分级 |
| 四 | 信号量与生产者-消费者 | 消息传递、生产者-消费者 | 信号量的计数与阻塞、流量控制 |
| 五 | 高级同步：条件变量·屏障·读写锁 | 生产者-消费者、归一化、并发链表 | 条件变量、屏障、读写锁 |
| 六 | 微架构感知：伪共享与无锁 | 生产者-消费者、**伪共享** | 缓存行、MESI、伪共享、原子操作 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">模块</th>
      <th style="text-align: left;">主题</th>
      <th style="text-align: left;">核心实验</th>
      <th style="text-align: left;">关键认识</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">一</td>
      <td style="text-align: left;">共享内存并行与 Pthread 基础</td>
      <td style="text-align: left;">Hello、矩阵向量乘</td>
      <td style="text-align: left;">Fork-Join、传参规范、公平的性能测量</td>
    </tr>
    <tr>
      <td style="text-align: left;">二</td>
      <td style="text-align: left;">数据竞争与临界区</td>
      <td style="text-align: left;">π 估算、消息传递</td>
      <td style="text-align: left;">数据竞争的定义、时序错误、<code>volatile</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">三</td>
      <td style="text-align: left;">互斥量：正确性与锁粒度</td>
      <td style="text-align: left;">π 估算、哲学家就餐</td>
      <td style="text-align: left;">最小化临界区、死锁四条件、资源分级</td>
    </tr>
    <tr>
      <td style="text-align: left;">四</td>
      <td style="text-align: left;">信号量与生产者-消费者</td>
      <td style="text-align: left;">消息传递、生产者-消费者</td>
      <td style="text-align: left;">信号量的计数与阻塞、流量控制</td>
    </tr>
    <tr>
      <td style="text-align: left;">五</td>
      <td style="text-align: left;">高级同步：条件变量·屏障·读写锁</td>
      <td style="text-align: left;">生产者-消费者、归一化、并发链表</td>
      <td style="text-align: left;">条件变量、屏障、读写锁</td>
    </tr>
    <tr>
      <td style="text-align: left;">六</td>
      <td style="text-align: left;">微架构感知：伪共享与无锁</td>
      <td style="text-align: left;">生产者-消费者、<strong>伪共享</strong></td>
      <td style="text-align: left;">缓存行、MESI、伪共享、原子操作</td>
    </tr>
  </tbody>
</table>

一条主线贯穿始终：

> **能并行**（会用线程）→ **正确并行**（消除竞争）→ **高效并行**（发挥多核效能）

三个阶段不可颠倒。本实验的伪共享，正是「高效并行」阶段最深、也最反直觉的一课——它提醒我们，并发程序的性能，最终要落实到硬件如何搬运数据这一层。

**先保证正确，再追求高效。** 这是本章九个实验反复印证的、也是并发编程最根本的方法论。

---

🎓 **第四章 Pthread 多线程编程 · 全部实验到此结束。**

从单个线程的创建，到多核缓存的一致性，我们完整地走过了共享内存并行编程的核心议题。掌握了这些原理与工具，你已经具备了阅读、编写和调试真实并发程序的基础。愿这条从「能用」到「用对」再到「用好」的路径，成为你今后深入并行计算领域的坚实起点。